# TNG × CMB Faraday rotation: photo-z / spec-z trade-off

**目的:** 同じTNG銀河母集団から、赤方偏移誤差と銀河数密度を独立に変え、
\(\langle\mathrm{RM}_{F}^{2}\delta_g(z_{\rm obs})\rangle\) の検出S/Nと
真の赤方偏移層ごとの振幅推定精度を比較する。符号付き \(B_{\rm FR}\times g\) は主推定量にしない。

この1ファイルをIllustrisTNG remote JupyterLabへアップロードし、**設定セルを確認してRun All**。
依存: numpy, scipy, pandas, matplotlib, h5py, astropy（CAMBは任意）。API key・ダウンロードは不要。

### この段階で計算するもの
- TNGガスからの符号付きRM視線積分、同じcone内の恒星質量選択銀河。
- contiguousな距離区間を箱の周期複製とnearest-full-snapshotで埋める**近似lightcone**。
- photo-zの散乱・平均バイアス・外れ値、spec-zの精度、共通母集団のランダム間引き。
- ビン混入行列、cross spectra、ビン間共分散、検出S/N、同時振幅推定、trade-off図。
- pure-E初期偏光 → exactなFR回転 → FR B-mode、multi-frequency Q/Uからの理想的なpixel RM復元。

### 解釈上の境界
1. これはまだ実サーベイの等級・色・観測成功率を再現するmockではない。
   photo/specというラベルは**同じ銀河母集団への測定精度・選択率**を意味する。
2. 前景は設定したz範囲だけ。CMB最終散乱面までの全磁場、再電離/再結合期の偏光生成は含めない。
3. 同じ箱を繰り返すので独立な広域実現ではない。広域S/NはGaussian共分散の面積換算であり、
   非Gaussian項・super-sample covariance・実際の天の川残差の空間構造を解決した予測ではない。
   FFTは正方形patchの周期近似。survey maskのmode couplingやapodizationは未処理。
4. 主解析のRM復元ノイズは**任意の感度パラメータ**。Q/Uのpixel復元は診断用で、EB quadratic estimatorではない。
   CMB実験の実性能を主解析のnoise値に対応づけていない。絶対的なCMB検出forecastと呼ばない。
5. default TNG300-3はI/O確認と探索用。低解像度の質量完全性、RM格子・cone投影の収束を確認し、
   本解析はTNG300-1/-2に変更する。ガス格子はセル中心でのNGP depositionで、Voronoi ray tracingではない。

参考: [Zhang & Lidz, arXiv:2512.06584](https://arxiv.org/html/2512.06584),
[Yadav et al., arXiv:1207.3356](https://arxiv.org/abs/1207.3356),
[TNG units/fields](https://www.tng-project.org/data/docs/specifications/).

In [ ]:
from pathlib import Path
import json, hashlib, warnings, time, importlib.util
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import h5py
from scipy.ndimage import map_coordinates
from scipy.integrate import cumulative_trapezoid
from scipy.special import ndtr
from astropy.cosmology import FlatLambdaCDM
from IPython.display import display
try:
    from tqdm.auto import tqdm
except ImportError:
    def tqdm(it, **kwargs): return it

plt.rcParams.update({'figure.dpi': 110, 'figure.figsize': (7, 4.5),
                     'axes.grid': True, 'grid.alpha': 0.2})

## 1. 設定 — 最初にここだけ変更

`MODE='tng'` が通常のremote実行。`MODE='synthetic'` はTNGファイルを必要としない動作検証用。
syntheticの結果はTNGやCMBの物理予測ではない。

defaultは高z側の `1.5 < z < 3` を3ビンで比較する。coneはphoto-z混入を受ける余白を含む。
`TRUE_Z_EDGES` を `[0.5, 1.0, 2.0, 3.0]` にすれば広いz範囲の比較になるが、読むsnapshotが増える。
最初のガス読込は数十GB以上になり得る。全ガスをRAMには載せない。cache生成後のパラメータ走査は軽い。

`RM_NOISE_RMS_GRID` は**平滑化前のpixelごとのRM雑音rms [rad/m²]**で、CMBのμK-arcminではない。
CMB再構成の感度が不明な段階で、銀河側の改善がどこで飽和するか調べるための軸。

In [ ]:
MODE = 'tng'                         # 'tng' or 'synthetic'
SIMULATION = 'TNG300-3'               # exploration; rerun with -2/-1 for convergence
BASE_PATH = None                     # e.g. Path('../sims.TNG/TNG300-3/output')
WORK_DIR = Path('fr_photoz_outputs')  # notebook execution directory, no simulation writes
# Support execution from either the repository root or the uploaded notebook folder.
NOTEBOOK_DIR = (Path.cwd() / 'detect-mag'
                if (Path.cwd() / 'detect-mag' / 'tng_cmb_fr_photoz_specz.ipynb').is_file()
                else Path.cwd())
CACHE_DIR = NOTEBOOK_DIR / 'cache'

TRUE_Z_EDGES = np.array([1.5, 2.0, 2.5, 3.0])
CONE_Z_RANGE = (0.8, 3.8)             # includes guard regions on both sides
MAP_N = 128
GRID_N = 128                         # 3 x GRID_N^3, float64 during accumulation
FOV_DEG = 1.5                       # flat-sky square; must fit in the TNG box at zmax
MIN_STELLAR_MASS_MSUN = 10**10.5
MIN_STAR_PARTICLES = 32              # resolution cut; physical sample changes if run changes
USE_RSD = True                      # z_spec baseline includes subhalo peculiar velocity
USE_CELL_HYDROGEN_FRACTION = True    # GFM_Metals[:,0], else fixed X_H=0.76
EXCLUDE_STAR_FORMING_GAS = False     # sensitivity check for effective multiphase ISM
READ_BATCH = 250_000                # bounded gas RAM; all files are read, no subsampling
SEED = 20261007

RM_SMOOTH_FWHM_ARCMIN = 3.0          # identical filter for every galaxy scenario
RM_NOISE_RMS_GRID = [0.0, 1.0, 10.0]
MW_RESIDUAL_RMS = 1.2               # white pixel proxy only; NOT a Milky Way sky model
EXTRA_PATH_RM_RMS = 0.0             # optional independent white proxy for omitted LoS
SURVEY_AREA_DEG2 = 1000.0           # Gaussian area extrapolation; NOT simulated sky area
N_BANDS = 10
ELL_MAX = 4000.0                    # actual cutoff also limited by map/grid resolution
Z_SIGMA0_GRID = [0.0, 0.01, 0.03, 0.05]  # sigma_z = sigma0 * (1+z_spec)
KEEP_FRACTIONS = [1.0, 0.3, 0.1]
PHOTO_Z_BIAS = 0.0                  # b_z * (1+z_spec)
PHOTO_OUTLIER_FRACTION = 0.0
SPEC_SIGMA0 = 0.0003
N_CATALOG_REALIZATIONS = 4          # conditional photo-z/thinning scatter, not cosmic variance

RUN_CMB_DEMO = True
CMB_FREQUENCIES_GHZ = np.array([30., 45., 90.])
CMB_NOISE_UK_ARCMIN = np.array([2., 2., 2.]) # illustrative inputs, NOT an experiment specification
CMB_BEAM_FWHM_ARCMIN = 3.0
CMB_E_MODEL = 'toy'                 # 'toy', 'camb', or 'file'
CMB_CL_FILE = None                  # npz with ell, cl_ee in microK^2; raw Cl, NOT Dl
INJECTION_AMPLITUDES = np.array([1.0, 1.0, 1.5])  # last/high-z bin excess; length must match bins

assert MODE in ('tng', 'synthetic')
assert len(TRUE_Z_EDGES) >= 3 and np.all(np.diff(TRUE_Z_EDGES) > 0)
assert CONE_Z_RANGE[0] < TRUE_Z_EDGES[0] < TRUE_Z_EDGES[-1] < CONE_Z_RANGE[1]
assert len(INJECTION_AMPLITUDES) == len(TRUE_Z_EDGES)-1
assert MAP_N >= 16 and GRID_N >= 8 and FOV_DEG > 0
assert SURVEY_AREA_DEG2 >= FOV_DEG**2 and SURVEY_AREA_DEG2 <= 41253
assert all(0 < f <= 1 for f in KEEP_FRACTIONS)
assert 0 <= PHOTO_OUTLIER_FRACTION < 1 and N_CATALOG_REALIZATIONS >= 1
assert all(s >= 0 for s in RM_NOISE_RMS_GRID) and MW_RESIDUAL_RMS >= 0
for p in (WORK_DIR, CACHE_DIR): p.mkdir(parents=True, exist_ok=True)

## 2. パス・cosmology・snapshotの選択

remoteの標準mountを探索する。見つからないときは`BASE_PATH`を指定する。
ガスの必要fieldが揃う**full snapshotのみ**を使い、coneの各tileの中央zに最も近いものを割り当てる。
nearest-snapshotの段階的な時間進化は近似で、snapshot間の滑らかなlightcone interpolationは行わない。

In [ ]:
FULL_SNAPS = [2, 3, 4, 6, 8, 11, 13, 17, 21, 25, 33, 40, 50, 59, 67, 72, 78, 84, 91, 99]

def find_base_path(sim, explicit=None):
    if explicit is not None:
        p = Path(explicit).expanduser().resolve()
        if not p.is_dir(): raise FileNotFoundError(f'BASE_PATH does not exist: {p}')
        return p
    candidates = []
    for root in [Path.cwd(), *list(Path.cwd().parents)[:4], Path.home(), Path('/home/jovyan')]:
        candidates += [root/'sims.TNG'/sim/'output', root/'sims.TNG'/sim,
                       root/'sims'/sim/'output']
    for p in candidates:
        if (p/'groups_099').is_dir(): return p.resolve()
    raise FileNotFoundError('TNG mount not found. Set BASE_PATH to sims.TNG/<run>/output.')

def group_files(base, snap):
    files = list((base/f'groups_{snap:03d}').glob(f'fof_subhalo_tab_{snap:03d}.*.hdf5'))
    return sorted(files, key=lambda p: int(p.name.split('.')[-2]))

def snapshot_files(base, snap):
    files = list((base/f'snapdir_{snap:03d}').glob(f'snap_{snap:03d}.*.hdf5'))
    return sorted(files, key=lambda p: int(p.name.split('.')[-2]))

def read_header(base, snap):
    files = group_files(base, snap)
    if not files: raise FileNotFoundError(f'No group catalog at snapshot {snap}')
    with h5py.File(files[0], 'r') as f: return dict(f['Header'].attrs)

if MODE == 'tng':
    BASE_PATH = find_base_path(SIMULATION, BASE_PATH)
    H = read_header(BASE_PATH, 99)
    h = float(H['HubbleParam']); box_mpc = float(H['BoxSize']) / (1000*h)
    cosmo = FlatLambdaCDM(H0=100*h, Om0=float(H['Omega0']))
    snap_headers = {s: read_header(BASE_PATH, s) for s in FULL_SNAPS}
    snap_z = {s: float(v['Redshift']) for s, v in snap_headers.items()}
    print('TNG path:', BASE_PATH)
else:
    h = 0.6774; box_mpc = 205/h
    cosmo = FlatLambdaCDM(H0=100*h, Om0=0.3089)
    snap_z = dict(zip([25,33,40,50,67], [3.,2.,1.5,1.,0.5]))

# Tabulate chi(z) and its inverse once. No repeated cosmology calls per gas cell/ray.
z_lookup = np.linspace(0, max(6., CONE_Z_RANGE[1]+1), 20001)
chi_lookup = cosmo.comoving_distance(z_lookup).value
z_to_chi = lambda z: np.interp(z, z_lookup, chi_lookup)
chi_to_z = lambda chi: np.interp(chi, chi_lookup, z_lookup)
theta_box = np.deg2rad(FOV_DEG)
assert z_to_chi(CONE_Z_RANGE[1])*theta_box < 0.9*box_mpc, (
    'FOV is too large for one transverse box at high z. Reduce FOV_DEG.')
chi_min, chi_max = z_to_chi(np.array(CONE_Z_RANGE))
n_tiles = int(np.ceil((chi_max-chi_min)/box_mpc))
chi_tile_edges = np.linspace(chi_min, chi_max, n_tiles+1)
geometry_rng = np.random.default_rng(SEED)
tiles = []
for t, (lo, hi) in enumerate(zip(chi_tile_edges[:-1], chi_tile_edges[1:])):
    zmid = float(chi_to_z((lo+hi)/2))
    snap = min(snap_z, key=lambda s: abs(snap_z[s]-zmid))
    # Cyclic permutation + an even number of sign flips is a proper cube rotation.
    perm = np.roll(np.arange(3), int(geometry_rng.integers(3)))
    signs = geometry_rng.choice([-1.,1.], size=3)
    signs[2] = signs[0]*signs[1]
    tiles.append(dict(tile=t, chi_lo=lo, chi_hi=hi, chi_mid=(lo+hi)/2,
                      zmid=zmid, snap=snap, snapshot_z=snap_z[snap],
                      perm=perm, signs=signs, offset=geometry_rng.uniform(0,box_mpc,3)))
display(pd.DataFrame([{k:v for k,v in t.items() if k not in ('perm','signs','offset')}
                      for t in tiles]))
print(f'Patch = {FOV_DEG**2:.3f} deg²; nominal survey = {SURVEY_AREA_DEG2:g} deg²')
print(f'Box = {box_mpc:.2f} cMpc; grid cell = {box_mpc/GRID_N:.2f} cMpc')
print('Repeated boxes are geometry surrogates, not independent cosmic realizations.')

## 3. ガスからRM積分用の格子を作る（重いセル・cacheあり）

TNG単位: \(B_{\rm phys}[G]=B_{\rm code}(h/a^2)\,2.60\times10^{-6}\)。
`MagneticField`にはさらに \(\sqrt{4\pi}\) を掛けない。
各セルの電子数 \(N_e=x_e X_H M/m_p\) と物理磁場の積を、comoving格子の体積で割ってdeposit:
\[
q_{c,k}=V_c^{-1}\sum_{\rm cells} N_e B_{{\rm phys},k}[\mu G]
\quad [\mathrm{cm}^{-3}\,\mu G].
\]
これはelectron-number-weightedな \(n_{e,c} B_{\rm phys}\) の体積平均。
セル中心NGP近似を採用し、電子密度とBを別々に平滑化してから掛ける方法とは違う。

\[
\mathrm{RM}_{\rm obs}=0.812\int
\frac{n_{e,\rm phys}[\mathrm{cm}^{-3}] B_{\parallel,\rm phys}[\mu G]}{(1+z)^2}
\,dl_{\rm proper}[\mathrm{pc}]
=0.812\int n_{e,c} B_{\parallel,\rm phys}\,d\chi[\mathrm{pc}].
\]
最後の等式は \(n_{e,\rm phys}=(1+z)^3 n_{e,c}\)、\(dl=d\chi/(1+z)\) による。
各tileではelectron number/comoving volumeを固定し、Bはnearest snapshotの物理値を使用。
star-forming gasのElectronAbundanceはeffective ISM model依存なので、除外の感度テストを用意する。

In [ ]:
MSUN_G = 1.98847e33
MPC_CM = 3.0856775814913673e24
MP_G = 1.67262192369e-24
B_UNIT_G = np.sqrt(1e10*MSUN_G/(MPC_CM/1000)) * 1e5/(MPC_CM/1000)
assert np.isclose(B_UNIT_G, 2.60e-6, rtol=0.01)
CACHE_VERSION = 'electron_B_NGP_v1'

# Metadata hashing prevents silent reuse after changing gas selection, units or resolution.
def cache_key(snap):
    meta = dict(version=CACHE_VERSION, sim=SIMULATION, path=str(BASE_PATH), snap=int(snap),
                grid=GRID_N, cell_XH=USE_CELL_HYDROGEN_FRACTION,
                exclude_SF=EXCLUDE_STAR_FORMING_GAS)
    return hashlib.sha256(json.dumps(meta, sort_keys=True).encode()).hexdigest()[:16]

def deposit_electron_B(pos_mpc, mass_code, xe, XH, B_code, h, a, box, ngrid):
    """Return extensive N_e B_phys per NGP voxel; input field vector stays signed."""
    ijk = np.floor((pos_mpc % box)/box*ngrid).astype(np.int64)
    idx = (ijk[:,0]*ngrid + ijk[:,1])*ngrid + ijk[:,2]
    electron_number = xe*XH*(mass_code*1e10*MSUN_G/h)/MP_G
    B_microG = B_code*(h/a**2)*B_UNIT_G*1e6
    return np.stack([np.bincount(idx, weights=electron_number*B_microG[:,k],
                                minlength=ngrid**3).reshape((ngrid,)*3) for k in range(3)])

def build_q_grid(snap):
    cache = CACHE_DIR/f'{SIMULATION}_snap{snap:03d}_{cache_key(snap)}.npz'
    # Reuse runs made before the cache directory was separated from outputs.
    legacy_cache = WORK_DIR / 'cache' / cache.name
    if not cache.exists() and legacy_cache.exists():
        import shutil
        shutil.copy2(legacy_cache, cache)
        print('Copied existing cache:', legacy_cache)
    if cache.exists():
        with np.load(cache) as f: q = f['q'].astype(np.float64)
        print('cache:', cache.name)
        return q
    files = snapshot_files(BASE_PATH, snap)
    if not files: raise FileNotFoundError(f'No gas snapshot files for {snap}')
    with h5py.File(files[0], 'r') as f:
        expected_files = int(f['Header'].attrs['NumFilesPerSnapshot'])
    if len(files) != expected_files:
        raise FileNotFoundError(f'Snapshot {snap}: found {len(files)} of '
                                f'{expected_files} chunks')
    a = 1/(1+snap_z[snap])
    sums = np.zeros((3, GRID_N, GRID_N, GRID_N), dtype=np.float64)
    read_cells = included_cells = fallback_XH = 0
    required = ['Coordinates','Masses','ElectronAbundance','MagneticField']
    started = time.time()
    for file in tqdm(files, desc=f'gas snapshot {snap}'):
        with h5py.File(file, 'r') as f:
            if 'PartType0' not in f: continue
            gas = f['PartType0']
            missing = set(required)-set(gas)
            if missing: raise KeyError(f'{file}: missing gas fields {missing}; use a full snapshot')
            if EXCLUDE_STAR_FORMING_GAS and 'StarFormationRate' not in gas:
                raise KeyError('StarFormationRate needed for the requested ISM cut')
            nc = gas['Coordinates'].shape[0]
            for start in range(0,nc,READ_BATCH):
                sl = slice(start,min(start+READ_BATCH,nc))
                pos = gas['Coordinates'][sl].astype(np.float64)/(1000*h)
                mass = gas['Masses'][sl].astype(np.float64)
                xe = gas['ElectronAbundance'][sl].astype(np.float64)
                bvec = gas['MagneticField'][sl].astype(np.float64)
                if USE_CELL_HYDROGEN_FRACTION and 'GFM_Metals' in gas:
                    xh = gas['GFM_Metals'][sl,0].astype(np.float64)
                else:
                    xh = np.full(len(mass),0.76); fallback_XH += len(mass)
                mask = (np.isfinite(pos).all(1) & np.isfinite(bvec).all(1)
                        & np.isfinite(mass) & (mass>0) & np.isfinite(xe) & (xe>=0)
                        & np.isfinite(xh) & (xh>0) & (xh<=1))
                if EXCLUDE_STAR_FORMING_GAS: mask &= gas['StarFormationRate'][sl] <= 0
                read_cells += len(mass); included_cells += int(mask.sum())
                sums += deposit_electron_B(pos[mask],mass[mask],xe[mask],xh[mask],
                                           bvec[mask],h,a,box_mpc,GRID_N)
    if included_cells == 0: raise ValueError(f'No usable gas cells at snapshot {snap}')
    q = sums / ((box_mpc/GRID_N*MPC_CM)**3)
    assert np.isfinite(q).all()
    np.savez_compressed(cache, q=q.astype(np.float32), snap=snap, z=snap_z[snap],
                        included_cells=included_cells, cache_version=CACHE_VERSION)
    print(f'snap {snap}: {included_cells:,}/{read_cells:,} gas cells, '
          f'{(time.time()-started)/60:.1f} min; XH fallback {fallback_XH:,}')
    return q

q_grids = {}
if MODE == 'tng':
    for snap in sorted(set(t['snap'] for t in tiles)):
        q_grids[snap] = build_q_grid(snap)
    print('Gas grid memory [MiB]:', sum(q.nbytes for q in q_grids.values())/2**20)

## 4. 同じconeへのガスと銀河の投影

tileごとに**ガス・銀河へ同じproper cube rotationと平行移動**を適用する。
LOS方向の反転時には磁場成分も符号反転する。銀河は恒星質量・star particle数・SubhaloFlagで選択。
個々の距離から \(z_{\rm cosm}\)、peculiar velocityから
\(z_{\rm spec}=z_{\rm cosm}+(1+z_{\rm cosm})v_\parallel/c\) を割り当てる。

coneの余白はprojectionにも含める。真の層分解はtargetビンに加えて**左右のguard層**を別成分として保持し、
振幅fitではguardの振幅を固定する（外からの漏れを無視しない）。
coneの外にある構造は生成していないので、その不足は余白拡大テストで確認する。

In [ ]:
N_TARGET = len(TRUE_Z_EDGES)-1
LAYER_EDGES = np.r_[CONE_Z_RANGE[0], TRUE_Z_EDGES, CONE_Z_RANGE[1]]
N_LAYERS = len(LAYER_EDGES)-1
TARGET_LAYERS = np.arange(1,1+N_TARGET)
GUARD_LAYERS = np.array([0,N_LAYERS-1])
ang_axis = (np.arange(MAP_N)+0.5)/MAP_N*theta_box-theta_box/2
AX, AY = np.meshgrid(ang_axis,ang_axis,indexing='ij')

# Interpolate periodic grid cell centers; coordinate 0 corresponds to the center of voxel 0.
def sample_los_q(q, tile, chi):
    global_coords = np.array([AX*chi,AY*chi,np.full_like(AX,chi-tile['chi_mid'])])
    original = np.empty_like(global_coords)
    for k in range(3):
        original[tile['perm'][k]] = tile['offset'][tile['perm'][k]] + tile['signs'][k]*global_coords[k]
    coords = original/box_mpc*GRID_N-0.5
    return tile['signs'][2]*map_coordinates(q[tile['perm'][2]], coords,
                                          order=1, mode='grid-wrap', prefilter=False)

def project_tile_rm(q, tile):
    result = np.zeros((N_LAYERS,MAP_N,MAP_N),dtype=np.float64)
    # Split exactly on every true layer boundary to avoid numerical leakage.
    cuts = np.unique(np.r_[tile['chi_lo'],tile['chi_hi'],
              z_to_chi(LAYER_EDGES)[(z_to_chi(LAYER_EDGES)>tile['chi_lo']) &
                                    (z_to_chi(LAYER_EDGES)<tile['chi_hi'])]])
    for lo,hi in zip(cuts[:-1],cuts[1:]):
        steps = max(1,int(np.ceil((hi-lo)/(0.5*box_mpc/GRID_N))))
        dchi = (hi-lo)/steps
        mid = float(chi_to_z((lo+hi)/2))
        layer = np.searchsorted(LAYER_EDGES,mid,side='right')-1
        for chi in lo+(np.arange(steps)+0.5)*dchi:
            result[layer] += 0.812*sample_los_q(q,tile,chi)*(dchi*1e6) # cMpc -> pc
    return result

def read_galaxy_parent(snap):
    # Catalogs are much smaller than gas, but still read chunk-by-chunk.
    pieces=[]; global_id=0
    files=group_files(BASE_PATH,snap)
    if not files: raise FileNotFoundError(f'No group catalog at snapshot {snap}')
    with h5py.File(files[0],'r') as f:
        expected=int(f['Header'].attrs.get('NumFiles',len(files)))
    if len(files)!=expected:
        raise FileNotFoundError(f'Snapshot {snap}: incomplete group catalog {len(files)}/{expected}')
    for file in files:
        with h5py.File(file,'r') as f:
            if 'Subhalo' not in f: continue
            g=f['Subhalo']; n=len(g['SubhaloPos'])
            mass=g['SubhaloMassType'][:,4].astype(float)*1e10/h
            counts=g['SubhaloLenType'][:,4]
            keep=(mass>=MIN_STELLAR_MASS_MSUN)&(counts>=MIN_STAR_PARTICLES)
            if 'SubhaloFlag' in g: keep &= g['SubhaloFlag'][:].astype(bool)
            if keep.any():
                pieces.append(dict(pos=g['SubhaloPos'][:][keep].astype(float)/(1000*h),
                                   vel=g['SubhaloVel'][:][keep].astype(float),
                                   mass=mass[keep], subhalo_id=np.arange(global_id,global_id+n)[keep]))
            global_id += n
    if not pieces: raise ValueError(f'No galaxies at snap {snap}; revise mass/resolution cut.')
    return {key:np.concatenate([p[key] for p in pieces]) for key in pieces[0]}

def project_tile_galaxies(parent,tile):
    d=(parent['pos']-tile['offset']+box_mpc/2)%box_mpc-box_mpc/2
    loc=d[:,tile['perm']]*tile['signs']
    chi=tile['chi_mid']+loc[:,2]
    keep=(chi>=tile['chi_lo'])&(chi<tile['chi_hi'])
    theta=loc[:,:2]/chi[:,None]
    keep &= np.all(np.abs(theta)<theta_box/2,axis=1)
    pix=np.floor((theta[keep]/theta_box+0.5)*MAP_N).astype(int)
    zcos=chi_to_z(chi[keep])
    vlos=tile['signs'][2]*parent['vel'][keep,tile['perm'][2]]
    zspec=zcos+(1+zcos)*vlos/299792.458 if USE_RSD else zcos.copy()
    return pd.DataFrame(dict(ix=pix[:,0],iy=pix[:,1],z_cos=zcos,z_spec=zspec,
                             stellar_mass=parent['mass'][keep],snap=tile['snap'],tile=tile['tile'],
                             subhalo_id=parent['subhalo_id'][keep]))

if MODE == 'tng':
    rm_layers=np.zeros((N_LAYERS,MAP_N,MAP_N))
    gal_pieces=[]
    for snap in sorted(q_grids):
        parent=read_galaxy_parent(snap)
        for tile in tqdm([t for t in tiles if t['snap']==snap],desc=f'cone snap {snap}'):
            rm_layers += project_tile_rm(q_grids[snap],tile)
            gal_pieces.append(project_tile_galaxies(parent,tile))
    galaxies=pd.concat(gal_pieces,ignore_index=True)
else:
    # Correlated lognormal galaxy intensity and signed magnetic amplitude for pipeline checks.
    from scipy.ndimage import gaussian_filter
    rng=np.random.default_rng(SEED)
    rm_layers=[]; gal_pieces=[]
    for j,(lo,hi) in enumerate(zip(LAYER_EDGES[:-1],LAYER_EDGES[1:])):
        density=gaussian_filter(rng.normal(size=(MAP_N,MAP_N)),3,mode='wrap')
        density=(density-density.mean())/density.std()
        intensity=np.exp(0.6*density-0.5*0.6**2)
        magnetic=gaussian_filter(rng.normal(size=(MAP_N,MAP_N)),0.7,mode='wrap')
        magnetic /= magnetic.std()
        rm_layers.append((1.5/(1+(lo+hi)/2))*intensity*magnetic)
        counts=rng.poisson(0.8*intensity)
        flat=np.repeat(np.arange(MAP_N**2),counts.ravel())
        zcos=rng.uniform(lo,hi,len(flat))
        zspec=zcos+(1+zcos)*rng.normal(0,150,len(flat))/299792.458 if USE_RSD else zcos
        gal_pieces.append(pd.DataFrame(dict(ix=flat//MAP_N,iy=flat%MAP_N,z_cos=zcos,
            z_spec=zspec,stellar_mass=np.full(len(flat),1e11),snap=-1,tile=j,
            subhalo_id=np.arange(len(flat)))))
    rm_layers=np.asarray(rm_layers); galaxies=pd.concat(gal_pieces,ignore_index=True)

galaxies['true_layer']=np.searchsorted(LAYER_EDGES,galaxies.z_cos,side='right')-1
assert len(galaxies)>0 and np.isfinite(rm_layers).all()
rm_total=rm_layers.sum(axis=0)             # signed sum FIRST
np.savez_compressed(WORK_DIR/'cone_rm_layers.npz',rm=rm_layers,layer_edges=LAYER_EDGES,
                    fov_deg=FOV_DEG,mode=MODE)
galaxies.to_csv(WORK_DIR/'cone_galaxies.csv',index=False)
display(galaxies.groupby('true_layer').agg(N=('ix','size'),zmin=('z_cos','min'),
                                         zmax=('z_cos','max'),mass_min=('stellar_mass','min')))
print('Galaxy count:',len(galaxies),'total RM rms:',np.std(rm_total),'rad/m²')
if MODE=='tng':
    # Free the large gas grids before parameter scans; caches are still on disk.
    q_grids.clear()

## 5. 角度filter、Fourier normalization、赤方偏移テンプレート

\(C_L=\Omega_{\rm pix}\,\Re(F_aF_b^*)/N_{\rm pix}\)（numpy FFTの規約）を環状平均。
monopoleは除外し、銀河場は \(N/\bar N-1\)、RM²場は**平均差し引きのみ**。
符号付きRMを平均RMで割らない。

\(R=\sum_jR_j\) を同じfilterで平滑化した後、\(X=R^2-\langle R^2\rangle\)。
真の層ごとの局所テンプレートを \(X_j=R R_j-\langle RR_j\rangle\) とする。
すると \(\sum_jX_j=X\) が実現ごとに厳密に成立し、異なる層間の交差項も消していない。
また \(R_j\rightarrow\sqrt{A_j}R_j\) としたとき、fiducial \(A_j=1\) で
\(\partial X/\partial A_j=X_j\)。従って以下の振幅fitは**磁場エネルギー振幅の局所線形化**。
大きい振幅変化に対する正確な非線形磁場fitではない。

\(y_{Li}=G_{Li}+\sum_jT_{Lij}A_j\)。左右guard層の \(G\) は固定し、target層の \(A_j\) を同時fit。
テンプレートはsimulation truthから校正するため、同じmockでの回収はclosure testであって独立な物理検証ではない。

In [ ]:
PIX_AREA_SR=(theta_box/MAP_N)**2
NPX=MAP_N**2
lx=2*np.pi*np.fft.fftfreq(MAP_N,d=theta_box/MAP_N)
LX,LY=np.meshgrid(lx,lx,indexing='ij')
LMAG=np.hypot(LX,LY)
sigma_theta=np.deg2rad(RM_SMOOTH_FWHM_ARCMIN/60)/np.sqrt(8*np.log(2))
RM_FILTER=np.exp(-0.5*LMAG**2*sigma_theta**2)

def filtered(field): return np.fft.ifft2(np.fft.fft2(field)*RM_FILTER).real
R_layers=np.array([filtered(r) for r in rm_layers])
R=R_layers.sum(0)
X=R**2-np.mean(R**2)
X_layers=R_layers*R[None,:,:]
X_layers -= X_layers.mean(axis=(1,2),keepdims=True)
assert np.allclose(X_layers.sum(0),X,rtol=1e-10,atol=1e-12*max(1.,np.max(np.abs(X))))

ell_min=2*np.pi/theta_box
# Conservative angular cutoff from largest physical grid cell / lowest cone distance.
grid_ell_limit=0.5*np.pi*z_to_chi(CONE_Z_RANGE[0])/(box_mpc/GRID_N)
ell_limit=min(ELL_MAX,0.65*np.pi*MAP_N/theta_box,grid_ell_limit)
if ell_limit <= 1.5*ell_min: raise ValueError('Too few resolved angular modes. Enlarge FOV or refine grid.')
ell_edges=np.geomspace(ell_min*0.99,ell_limit,N_BANDS+1)
mode_masks=[(LMAG>=a)&(LMAG<b) for a,b in zip(ell_edges[:-1],ell_edges[1:])]
mode_masks=[m for m in mode_masks if np.count_nonzero(m)>=4]
if len(mode_masks)<2: raise ValueError('Fewer than two populated Fourier bands.')
ell=np.array([LMAG[m].mean() for m in mode_masks])
patch_modes=np.array([m.sum() for m in mode_masks]) # includes +/-; standard Gaussian 2 for auto
area_ratio=SURVEY_AREA_DEG2/FOV_DEG**2
forecast_modes=patch_modes*area_ratio

def bin_fourier(values): return np.array([np.mean(values[m]) for m in mode_masks])
def spectrum(a,b):
    return bin_fourier(PIX_AREA_SR/NPX*(np.fft.fft2(a)*np.fft.fft2(b).conj()).real)

def catalog_maps(zobs,keep):
    obsbin=np.searchsorted(TRUE_Z_EDGES,zobs,side='right')-1
    maps=[]; counts=[]
    for i in range(N_TARGET):
        use=keep&(obsbin==i)
        cnt=np.bincount((galaxies.ix.values[use]*MAP_N+galaxies.iy.values[use]),
                        minlength=NPX).reshape(MAP_N,MAP_N)
        if cnt.sum()<10:
            raise ValueError(f'Only {cnt.sum()} galaxies in observed bin {i}; '
                             'increase retention/FOV or revise mass cuts/redshift bins.')
        maps.append(cnt/cnt.mean()-1);counts.append(int(cnt.sum()))
    return np.array(maps),np.array(counts),obsbin

fig,axes=plt.subplots(1,3,figsize=(15,4))
for ax,field,title in zip(axes,[R,X,np.log1p(np.bincount(
        galaxies.ix.values*MAP_N+galaxies.iy.values,minlength=NPX).reshape(MAP_N,MAP_N))],
        ['smoothed signed RM [rad/m²]','centered RM² [rad²/m⁴]','log(1 + galaxy count)']):
    im=ax.imshow(field.T,origin='lower',extent=[-FOV_DEG/2,FOV_DEG/2]*2)
    ax.set_title(title);ax.set_xlabel('angle [deg]');plt.colorbar(im,ax=ax)
plt.tight_layout();plt.savefig(WORK_DIR/'cone_maps.png',dpi=150);plt.show()
print('Usable L bands:',ell.round(),'patch mode counts:',patch_modes)

## 6. photo-z/spec-z実験とGaussian共分散

観測誤差は \(z_{\rm obs}=z_{\rm spec}+b_z(1+z_{\rm spec})+
\sigma_0(1+z_{\rm spec})\epsilon\)。外れ値はconeのz範囲に一様再配置。
共通の乱数を使い、保持率だけを変えた比較はnestedな銀河サンプルにする。

同じRM²マップを全zビンで使うので、ビン間共分散を保持する。
Gaussian近似でbandごとに
\[
\mathrm{Cov}_{L,ij}=
\frac{C_L^{XX,\rm total} C_L^{g_i g_j,\rm total}
+C_L^{Xg_i}C_L^{Xg_j}}{N_{\rm modes,L}}.
\]
信号テンプレートはphoto-z PDFから作る**期待銀河map**で校正し、間引きの偶然の相関を信号と数えない。
銀河autoは期待mapのdiscretenessに、categoricalなビン割当て・間引きの追加共分散を足す。
既に含まれる親catalogのshot noiseを二重計上しない。
信号のsample varianceをこの近似の中では残す。connected非Gaussian covarianceは未計算。

RMの白色pixel雑音Nをfilterに通すと、\(\xi_N=\sigma_N^2\,\mathrm{IFFT}(|F|^2)\)。
二乗場の雑音powerには \(2RN\) も入り、\(\xi_{X,\rm noise}=4\xi_R\xi_N+2\xi_N^2\)。
単なる「RM雑音を二乗したwhite noise」にはしない。均一noise biasのmonopoleは平均差し引きで消える。

In [ ]:
def make_observation(sigma0,keep_fraction,seed,bias=0.,outlier_fraction=0.,cosmological=False):
    rng=np.random.default_rng(seed)
    zbase=galaxies.z_cos.values if cosmological else galaxies.z_spec.values
    gaussian=rng.normal(size=len(galaxies))
    keep_uniform=rng.random(len(galaxies))
    catastrophic=rng.random(len(galaxies))<outlier_fraction
    random_z=rng.uniform(*CONE_Z_RANGE,len(galaxies))
    zobs=zbase+(1+zbase)*(bias+sigma0*gaussian)
    zobs[catastrophic]=random_z[catastrophic]
    keep=keep_uniform<keep_fraction
    return zobs,keep

def mixing_diagnostics(obsbin,keep):
    n=np.zeros((N_TARGET,N_LAYERS),dtype=int)
    for i in range(N_TARGET):
        for j in range(N_LAYERS):
            n[i,j]=np.count_nonzero(keep&(obsbin==i)&(galaxies.true_layer.values==j))
    # P(true layer | observed bin), rows sum to one including guards.
    contamination=n/np.maximum(n.sum(1,keepdims=True),1)
    # P(observed target bin | true layer, retained); rows need not sum to one after losses.
    totals=np.bincount(galaxies.true_layer.values[keep],minlength=N_LAYERS)
    transfer=(n/np.maximum(totals[None,:],1)).T
    return contamination,transfer

def squared_noise_power(rm_noise_rms):
    sigma2=rm_noise_rms**2+MW_RESIDUAL_RMS**2+EXTRA_PATH_RM_RMS**2
    xi_r=np.fft.ifft2(np.abs(np.fft.fft2(R))**2).real/NPX
    xi_n=sigma2*np.fft.ifft2(RM_FILTER**2).real
    noise2d=PIX_AREA_SR*np.fft.fft2(4*xi_r*xi_n+2*xi_n**2).real
    # Tiny negative roundoff is not physical; larger negatives signal an implementation issue.
    if noise2d.min() < -1e-8*max(1.,noise2d.max()):
        raise ValueError('Negative squared-noise power beyond numerical roundoff')
    return np.maximum(bin_fourier(noise2d),0.)

C_XX=spectrum(X,X)

def expected_galaxy_maps(sigma0,keep_fraction,bias,outliers,cosmological):
    # Conditional ensemble over measurement errors and thinning: do not use a noisy
    # thinned realization to calibrate the forecast signal itself.
    zbase=galaxies.z_cos.values if cosmological else galaxies.z_spec.values
    mean=zbase+(1+zbase)*bias
    std=sigma0*(1+zbase)
    if sigma0>0:
        prob=ndtr((TRUE_Z_EDGES[None,1:]-mean[:,None])/std[:,None])
        prob-=ndtr((TRUE_Z_EDGES[None,:-1]-mean[:,None])/std[:,None])
    else:
        prob=((mean[:,None]>=TRUE_Z_EDGES[None,:-1])&
              (mean[:,None]<TRUE_Z_EDGES[None,1:])).astype(float)
    uniform_prob=np.diff(TRUE_Z_EDGES)/(CONE_Z_RANGE[1]-CONE_Z_RANGE[0])
    prob=(1-outliers)*prob+outliers*uniform_prob[None,:]
    expected_n=prob.sum(0)
    if np.any(expected_n*keep_fraction<10):
        raise ValueError('Expected retained count < 10 in a z bin; revise cuts/density.')
    pixels=galaxies.ix.values*MAP_N+galaxies.iy.values
    maps=[]
    for i in range(N_TARGET):
        cnt=np.bincount(pixels,weights=prob[:,i],minlength=NPX).reshape(MAP_N,MAP_N)
        maps.append(cnt/(expected_n[i]/NPX)-1)
    # E[Cgg of the categorical photo-z/thinning count maps | parent].
    # The parent weighted map already contains its own discreteness. Add only
    # the extra assignment/thinning variance, including off-diagonal terms.
    extra=theta_box**2*(np.diag(1/(keep_fraction*expected_n))-
                       (prob.T@prob)/(expected_n[:,None]*expected_n[None,:]))
    return np.array(maps),extra,expected_n*keep_fraction

def covariance_blocks(gmaps,signal,rm_noise_rms,extra_galaxy_noise=None):
    cgg=np.empty((len(ell),N_TARGET,N_TARGET))
    for i in range(N_TARGET):
        for j in range(N_TARGET): cgg[:,i,j]=spectrum(gmaps[i],gmaps[j])
    if extra_galaxy_noise is not None: cgg += extra_galaxy_noise[None,:,:]
    cxx=C_XX+squared_noise_power(rm_noise_rms)
    blocks=[]
    for b in range(len(ell)):
        cov=(cxx[b]*cgg[b]+np.outer(signal[b],signal[b]))/forecast_modes[b]
        # Stabilize only machine-scale degeneracy, not missing information.
        cov=(cov+cov.T)/2
        eig,vec=np.linalg.eigh(cov)
        floor=max(float(eig.max()),1e-300)*1e-10
        cov=(vec*np.maximum(eig,floor))@vec.T
        blocks.append(cov)
    return np.array(blocks)

def template_fit(signal,templates,guard,cov,injected):
    """Band-major covariance; target energy amplitudes, guard held fixed."""
    F=np.zeros((N_TARGET,N_TARGET));rhs=np.zeros(N_TARGET);snr2=0.
    y=signal-guard
    for b in range(len(ell)):
        inv=np.linalg.inv(cov[b]);T=templates[b]
        F += T.T@inv@T
        rhs += T.T@inv@y[b]
        snr2 += signal[b]@inv@signal[b]
    scale=np.sqrt(np.maximum(np.diag(F),1e-300))
    Fc=F/scale[:,None]/scale[None,:]
    eig=np.linalg.eigvalsh(Fc)
    full_rank=bool(eig.min()>1e-8*max(eig.max(),1e-300))
    if full_rank:
        acov=np.linalg.inv(Fc)/scale[:,None]/scale[None,:]
        A=acov@rhs
        error=np.sqrt(np.diag(acov))
        injected_rhs=np.zeros(N_TARGET)
        for b in range(len(ell)):
            injected_rhs += templates[b].T@np.linalg.solve(cov[b],templates[b]@injected)
        recovered=acov@injected_rhs
    else:
        acov=np.full_like(F,np.nan);A=error=recovered=np.full(N_TARGET,np.nan)
    per_bin=np.sqrt(np.sum(signal**2/np.diagonal(cov,axis1=1,axis2=2),axis=0))
    return dict(snr_joint=np.sqrt(max(snr2,0.)),snr_bins=per_bin,
                A=A,error=error,amplitude_cov=acov,recovered_injection=recovered,
                full_rank=full_rank,condition=np.linalg.cond(Fc))

def analyze_scenario(sigma0,keep_fraction,rm_noise_rms,seed,bias=0.,outliers=0.,cosmological=False):
    zobs,keep=make_observation(sigma0,keep_fraction,seed,bias,outliers,cosmological)
    drawn_gmaps,counts,obsbin=catalog_maps(zobs,keep)
    gmaps,extra_gnoise,expected_counts=expected_galaxy_maps(
        sigma0,keep_fraction,bias,outliers,cosmological)
    # T[band, observed bin, true target layer]; guard includes both exterior layers.
    all_templates=np.array([[spectrum(xj,gi) for xj in X_layers] for gi in gmaps])
    all_templates=all_templates.transpose(2,0,1)
    T=all_templates[:,:,TARGET_LAYERS]
    G=all_templates[:,:,GUARD_LAYERS].sum(2)
    signal=np.array([spectrum(X,gi) for gi in gmaps]).T
    assert np.allclose(signal,G+T.sum(2),rtol=1e-9,atol=1e-12*max(1.,np.max(abs(signal))))
    cov=covariance_blocks(gmaps,signal,rm_noise_rms,extra_gnoise)
    fit=template_fit(signal,T,G,cov,INJECTION_AMPLITUDES)
    drawn_signal=np.array([spectrum(X,gi) for gi in drawn_gmaps]).T
    catalog_fit=template_fit(drawn_signal,T,G,cov,INJECTION_AMPLITUDES)
    mix,transfer=mixing_diagnostics(obsbin,keep)
    # Fraction outside cone expected for retained population under non-catastrophic Gaussian errors.
    zbase=galaxies.z_cos.values if cosmological else galaxies.z_spec.values
    std=sigma0*(1+zbase)
    mean=zbase+bias*(1+zbase)
    if sigma0>0:
        p_out=ndtr((CONE_Z_RANGE[0]-mean)/std)+1-ndtr((CONE_Z_RANGE[1]-mean)/std)
    else: p_out=((mean<CONE_Z_RANGE[0])|(mean>=CONE_Z_RANGE[1])).astype(float)
    return dict(fit=fit,signal=signal,templates=T,guard=G,cov=cov,counts=counts,
                contamination=mix,transfer=transfer,gmaps=gmaps,expected_counts=expected_counts,
                drawn_signal=drawn_signal,catalog_A=catalog_fit['A'],
                mean_outside_cone_probability=float(np.mean(p_out[keep])))

## 7. 全パラメータ走査と保存

truthはz_cosによる理想基準。controlled/specはpeculiar velocity込みのz_specを基準にする。
基準・photo-z scatter・spec-z精度を同じ母集団/面積で比較。
保持率を下げるだけなら銀河バイアスはensemble平均で不変。
catalog realizationsはphoto-z測定/間引きの条件付き乱数を変えるだけで、独立なTNG universeではない。
結果表の`A_error`はGaussian forecast、`A_closure`はtemplateの自己整合性。
`A_catalog_realization`/`A_catalog_scatter`は実際に散乱・間引いたcatalogの条件付きばらつき。
主signalは期待PDFで校正するので、間引きにより偶然cross signalが増える効果をforecastと混同しない。
rank不足では誤差をNaNとし、擬似逆行列で見かけの精度を作らない。

In [ ]:
rows=[]; failures=[]; representative={}
scenarios=[('truth',0.,1.,0.,0.)]
scenarios += [('controlled',s,f,PHOTO_Z_BIAS,PHOTO_OUTLIER_FRACTION)
              for s in Z_SIGMA0_GRID for f in KEEP_FRACTIONS]
scenarios += [('spec',SPEC_SIGMA0,f,0.,0.) for f in KEEP_FRACTIONS]
for family,sigma0,keep_fraction,bias,outliers in tqdm(scenarios,desc='galaxy scenarios'):
    for k in range(N_CATALOG_REALIZATIONS):
        seed=SEED+1000+k
        for rm_noise in RM_NOISE_RMS_GRID:
            try:
                result=analyze_scenario(sigma0,keep_fraction,rm_noise,seed,bias,outliers,
                                        cosmological=(family=='truth'))
            except ValueError as e:
                if family=='truth':
                    raise RuntimeError('Baseline sample is too sparse for the selected bins. '
                        'Increase FOV within the box limit, lower a justified mass cut, '
                        'or use a higher-resolution simulation.') from e
                failures.append(dict(family=family,sigma0=sigma0,keep=keep_fraction,
                                     rm_noise=rm_noise,realization=k,reason=str(e)))
                continue
            fit=result['fit']
            key=(family,float(sigma0),float(keep_fraction),float(rm_noise))
            if k==0: representative[key]=result
            for i in range(N_TARGET):
                rows.append(dict(family=family,sigma0=sigma0,keep_fraction=keep_fraction,
                    rm_noise_rms=rm_noise,realization=k,z_bin=i,z_lo=TRUE_Z_EDGES[i],
                    z_hi=TRUE_Z_EDGES[i+1],Ngal_patch=result['counts'][i],
                    n_deg2=result['counts'][i]/FOV_DEG**2,
                    expected_n_deg2=result['expected_counts'][i]/FOV_DEG**2,
                    A_catalog_realization=result['catalog_A'][i],
                    snr_bin=fit['snr_bins'][i],snr_joint=fit['snr_joint'],
                    A_error=fit['error'][i],A_closure=fit['A'][i],
                    A_injected=INJECTION_AMPLITUDES[i],
                    A_injection_recovered=fit['recovered_injection'][i],
                    full_rank=fit['full_rank'],fisher_condition=fit['condition'],
                    guard_contamination=result['contamination'][i,GUARD_LAYERS].sum(),
                    mean_outside_cone_probability=result['mean_outside_cone_probability']))
results=pd.DataFrame(rows)
if results.empty:
    raise RuntimeError('No usable scenario. See failures: '+str(failures[:3]))
results.to_csv(WORK_DIR/'tradeoff_all_realizations.csv',index=False)
summary=(results.groupby(['family','sigma0','keep_fraction','rm_noise_rms','z_bin'])
    .agg(n_deg2=('n_deg2','mean'),expected_n_deg2=('expected_n_deg2','mean'),
         snr_bin=('snr_bin','mean'),snr_joint=('snr_joint','mean'),
         A_error=('A_error','mean'),A_error_scatter=('A_error','std'),
         A_catalog_scatter=('A_catalog_realization','std'),
         A_closure=('A_closure','mean'),guard_contamination=('guard_contamination','mean'),
         full_rank_fraction=('full_rank','mean')).reset_index())
summary.to_csv(WORK_DIR/'tradeoff_summary.csv',index=False)
if failures:
    pd.DataFrame(failures).to_csv(WORK_DIR/'failed_scenarios.csv',index=False)
    warnings.warn(f'{len(failures)} scenarios failed, usually too few galaxies; see failed_scenarios.csv')
display(summary.head(15))
print('Largest fiducial closure error:',np.nanmax(np.abs(results.A_closure-1)))
print('Largest injected closure error:',np.nanmax(np.abs(results.A_injection_recovered-results.A_injected)))

## 8. trade-off可視化: 検出S/Nと赤方偏移分解を別々に見る

各panelの数字は同じ任意RM雑音・同じsurvey areaでの条件付きGaussian forecast。
幅広いbinの高い検出S/Nが、狭い真のz層の振幅を高精度で測れることを意味するとは限らない。
この図は実サーベイ間の優劣ではなく、同じ銀河母集団に対する数密度とz精度の比較。

In [ ]:
for noise in RM_NOISE_RMS_GRID:
    table=summary[(summary.family=='controlled')&(summary.rm_noise_rms==noise)]
    fig,axes=plt.subplots(2,N_TARGET,figsize=(5*N_TARGET,8),squeeze=False)
    for i in range(N_TARGET):
        sub=table[table.z_bin==i]
        for row,metric in enumerate(['snr_bin','A_error']):
            pivot=sub.pivot(index='keep_fraction',columns='sigma0',values=metric)
            pivot=pivot.reindex(index=sorted(KEEP_FRACTIONS),columns=Z_SIGMA0_GRID)
            values=pivot.values
            im=axes[row,i].imshow(values,origin='lower',aspect='auto',cmap='viridis')
            axes[row,i].set_xticks(range(len(pivot.columns)))
            axes[row,i].set_xticklabels([f'{v:g}' for v in pivot.columns])
            axes[row,i].set_yticks(range(len(pivot.index)))
            axes[row,i].set_yticklabels([f'{v:g}' for v in pivot.index])
            axes[row,i].set_xlabel('sigma_z / (1+z)');axes[row,i].set_ylabel('retained fraction')
            axes[row,i].set_title(f'{TRUE_Z_EDGES[i]:g}<z<{TRUE_Z_EDGES[i+1]:g}: {metric}')
            for yy in range(values.shape[0]):
                for xx in range(values.shape[1]):
                    axes[row,i].text(xx,yy,f'{values[yy,xx]:.2g}',ha='center',va='center',color='white')
            plt.colorbar(im,ax=axes[row,i])
    fig.suptitle(f'RM noise before filter={noise:g} rad/m²; MW proxy={MW_RESIDUAL_RMS:g}; '
                 f'Gaussian area={SURVEY_AREA_DEG2:g} deg²; {MODE}')
    plt.tight_layout();plt.savefig(WORK_DIR/f'tradeoff_noise_{noise:g}.png',dpi=150);plt.show()

# Direct photo/spec comparison at a common RM sensitivity, with density on the x-axis.
noise=float(RM_NOISE_RMS_GRID[min(1,len(RM_NOISE_RMS_GRID)-1)])
photo_sigma=float(Z_SIGMA0_GRID[min(2,len(Z_SIGMA0_GRID)-1)])
fig,axes=plt.subplots(2,N_TARGET,figsize=(5*N_TARGET,8),squeeze=False)
for i in range(N_TARGET):
    for family,sig,label in [('controlled',photo_sigma,f'photo sigma0={photo_sigma:g}'),
                              ('spec',SPEC_SIGMA0,f'spec sigma0={SPEC_SIGMA0:g}')]:
        sub=summary[(summary.family==family)&(summary.sigma0==sig)&
                    (summary.rm_noise_rms==noise)&(summary.z_bin==i)].sort_values('expected_n_deg2')
        for row,metric in enumerate(['snr_bin','A_error']):
            axes[row,i].plot(sub.expected_n_deg2,sub[metric],'o-',label=label)
            axes[row,i].set_xscale('log');axes[row,i].set_yscale('log')
            axes[row,i].set_xlabel('expected selected galaxies / deg²')
            axes[row,i].set_ylabel(metric);axes[row,i].legend(fontsize=8)
            axes[row,i].set_title(f'{TRUE_Z_EDGES[i]:g}<z<{TRUE_Z_EDGES[i+1]:g}')
fig.suptitle(f'Controlled parent population; RM noise={noise:g} rad/m²; {MODE}')
plt.tight_layout();plt.savefig(WORK_DIR/'photo_spec_comparison.png',dpi=150);plt.show()

## 9. ビン混入・cross spectra・振幅相関

representativeなphoto-z caseで、guard由来の寄与まで表示。
続いて、混入を無視したテンプレートでfitしたときのバイアスを比較する。
この比較は今回のcone実現に条件付けたもの。

In [ ]:
ref_noise=float(RM_NOISE_RMS_GRID[min(1,len(RM_NOISE_RMS_GRID)-1)])
photo_sigma=float(Z_SIGMA0_GRID[-1])
photo_key=('controlled',photo_sigma,1.0,ref_noise)
if photo_key not in representative:
    photo_key=next(k for k in representative if k[0]=='controlled')
rphoto=representative[photo_key]
rtruth=representative[('truth',0.,1.,ref_noise)]
fig,axes=plt.subplots(1,3,figsize=(17,4.5))
im=axes[0].imshow(rphoto['contamination'],vmin=0,vmax=1,aspect='auto')
axes[0].set_xlabel('true layer (guards at both ends)');axes[0].set_ylabel('observed z bin')
axes[0].set_title('P(true layer | observed bin)')
axes[0].set_xticks(range(N_LAYERS))
axes[0].set_xticklabels([f'{a:g}–{b:g}' for a,b in zip(LAYER_EDGES[:-1],LAYER_EDGES[1:])],rotation=35)
axes[0].set_yticks(range(N_TARGET))
axes[0].set_yticklabels([f'{a:g}–{b:g}' for a,b in zip(TRUE_Z_EDGES[:-1],TRUE_Z_EDGES[1:])])
for i in range(N_TARGET):
    for j in range(N_LAYERS): axes[0].text(j,i,f'{rphoto["contamination"][i,j]:.2f}',ha='center',color='white')
plt.colorbar(im,ax=axes[0])
for i in range(N_TARGET):
    axes[1].plot(ell,rtruth['signal'][:,i],label=f'truth bin {i}')
    axes[1].plot(ell,rphoto['signal'][:,i],'--',label=f'photo bin {i}')
axes[1].set_xscale('log');axes[1].set_yscale('symlog',linthresh=max(np.max(abs(rtruth['signal']))*0.01,1e-20))
axes[1].set_xlabel('L');axes[1].set_ylabel('C_L(RM², g) [rad²/m⁴ × sr]');axes[1].legend(fontsize=8)
acov=rphoto['fit']['amplitude_cov']
if np.isfinite(acov).all():
    corr=acov/np.sqrt(np.diag(acov)[:,None]*np.diag(acov)[None,:])
    im=axes[2].imshow(corr,vmin=-1,vmax=1,cmap='coolwarm');plt.colorbar(im,ax=axes[2])
    axes[2].set_title('Correlation of fitted true-z amplitudes')
else: axes[2].text(0.1,0.5,'Rank deficient: amplitudes not identifiable')
plt.tight_layout();plt.savefig(WORK_DIR/'redshift_mixing_and_spectra.png',dpi=150);plt.show()

# Use common full-density parent, but intentionally calibrate against no-photo-z templates.
yinj=rphoto['guard']+np.einsum('bij,j->bi',rphoto['templates'],INJECTION_AMPLITUDES)
wrong=template_fit(yinj,rtruth['templates'],rtruth['guard'],rphoto['cov'],INJECTION_AMPLITUDES)
correct=template_fit(yinj,rphoto['templates'],rphoto['guard'],rphoto['cov'],INJECTION_AMPLITUDES)
bias_table=pd.DataFrame(dict(z_lo=TRUE_Z_EDGES[:-1],z_hi=TRUE_Z_EDGES[1:],
    injected=INJECTION_AMPLITUDES,correct_mixing_fit=correct['A'],
    ignored_mixing_fit=wrong['A'],forecast_error=correct['error']))
display(bias_table);bias_table.to_csv(WORK_DIR/'mixing_bias_demo.csv',index=False)
np.savez_compressed(WORK_DIR/'representative_templates.npz',ell=ell,
    templates=rphoto['templates'],guard=rphoto['guard'],cov=rphoto['cov'],
    contamination=rphoto['contamination'],transfer=rphoto['transfer'],
    true_z_edges=TRUE_Z_EDGES,layer_edges=LAYER_EDGES)

## 10. CMB pure-E → FR → B-mode（診断用）

初期E-modeはTNGと独立に生成し、初期B=0。
`toy`はrms=1 μKに規格化した解析的なE spectrumで、宇宙論的な検出予測には使わない。
`camb`はインストール済みCAMBでunlensed scalar EEを生成する（ここから自動installはしない）。
`file`なら `ell, cl_ee` を持つnpzを指定する。**Cl**を入れ、Dlは事前に変換する。

回転規約は \(P_\nu=(Q+iU)_\nu=P_0\exp(2i\lambda_\nu^2 R)\)。
exact回転を行い、FFTのspin-2変換でE/Bに戻す。
本cellではlens B-mode、天の川回転、instrument noiseをまだ加えず、磁場由来の変化だけを確認する。
実行中のcone範囲にあるpost-recombination screenだけをモデル化している。

In [ ]:
def make_cmb_ee():
    if CMB_E_MODEL=='file':
        if CMB_CL_FILE is None: raise ValueError('Set CMB_CL_FILE for file mode')
        with np.load(CMB_CL_FILE) as f: ls=f['ell'];cls=f['cl_ee']
        if np.any(cls<0): raise ValueError('EE Cl must be nonnegative')
        if np.max(ls)<LMAG.max(): warnings.warn('EE file does not cover map Nyquist; uncovered modes set to zero')
        return np.interp(LMAG,ls,cls,left=0,right=0),'input EE Cl'
    if CMB_E_MODEL=='camb':
        try: import camb
        except ImportError as e: raise ImportError('CAMB unavailable: select toy or provide a Cl file') from e
        lmax=int(np.ceil(LMAG.max()))+100
        pars=camb.CAMBparams();pars.set_cosmology(H0=100*h,ombh2=0.0223,
              omch2=float(cosmo.Om0)*h*h-0.0223,mnu=0.06,tau=0.054)
        pars.InitPower.set_params(As=2.1e-9,ns=0.9667);pars.set_for_lmax(lmax,lens_potential_accuracy=0)
        cls=camb.get_results(pars).get_cmb_power_spectra(pars,CMB_unit='muK',raw_cl=True)['unlensed_scalar'][:,1]
        return np.interp(LMAG,np.arange(len(cls)),cls,left=0,right=0),'CAMB unlensed scalar EE'
    if CMB_E_MODEL!='toy': raise ValueError('Unknown CMB_E_MODEL')
    shape=np.exp(-(LMAG/3000)**2)*(1+0.4*np.cos(LMAG/120))**2/(LMAG**2+100**2)
    shape[0,0]=0
    return shape,'TOY EE (rms normalized, not a physical forecast)'

cos2phi=np.cos(2*np.arctan2(LY,LX));sin2phi=np.sin(2*np.arctan2(LY,LX))
# Remove Nyquist row/column: spin kernels at these self-conjugate modes need special treatment.
spin_mask=np.ones((MAP_N,MAP_N));spin_mask[0,0]=0
if MAP_N%2==0: spin_mask[MAP_N//2,:]=0;spin_mask[:,MAP_N//2]=0

def eb_to_qu(Ek,Bk):
    return (np.fft.ifft2((cos2phi*Ek-sin2phi*Bk)*spin_mask).real,
            np.fft.ifft2((sin2phi*Ek+cos2phi*Bk)*spin_mask).real)

def qu_to_eb(Q,U):
    Qk,Uk=np.fft.fft2(Q),np.fft.fft2(U)
    return ((cos2phi*Qk+sin2phi*Uk)*spin_mask,
            (-sin2phi*Qk+cos2phi*Uk)*spin_mask)

if RUN_CMB_DEMO:
    ee_cl,cmb_label=make_cmb_ee()
    rng=np.random.default_rng(SEED+700)
    Ek=np.fft.fft2(rng.normal(size=(MAP_N,MAP_N)))*np.sqrt(ee_cl/PIX_AREA_SR)*spin_mask
    if CMB_E_MODEL=='toy':
        Ek /= np.std(np.fft.ifft2(Ek).real)
    Q0,U0=eb_to_qu(Ek,np.zeros_like(Ek));P0=Q0+1j*U0
    frequencies=CMB_FREQUENCIES_GHZ
    lambda2=(0.299792458/frequencies)**2 # c [m GHz] / frequency [GHz]
    P_rot=np.exp(2j*lambda2[:,None,None]*R[None,:,:])*P0[None,:,:]
    B_FR=np.array([np.fft.ifft2(qu_to_eb(p.real,p.imag)[1]).real for p in P_rot])
    assert np.max(abs(qu_to_eb(Q0,U0)[1])) < 1e-9*max(1.,np.max(abs(Ek)))
    fig,axes=plt.subplots(1,2,figsize=(12,4))
    im=axes[0].imshow(B_FR[0].T,origin='lower');plt.colorbar(im,ax=axes[0],label='microK')
    axes[0].set_title(f'FR B-mode at {frequencies[0]:g} GHz; {cmb_label}')
    gtrue=rtruth['gmaps'][0]
    for bfield,nu in zip(B_FR,frequencies):
        axes[1].plot(ell,spectrum(bfield**2-np.mean(bfield**2),gtrue),label=f'B_FR² × g, {nu:g} GHz')
    axes[1].set_xscale('log');axes[1].set_yscale('symlog',linthresh=1e-16)
    axes[1].set_xlabel('L');axes[1].set_ylabel('C_L(B_FR², g) [microK² × sr]');axes[1].legend()
    plt.tight_layout();plt.savefig(WORK_DIR/'cmb_fr_bmode.png',dpi=150);plt.show()
    np.savez_compressed(WORK_DIR/'cmb_fr_demo.npz',Q_ini=Q0,U_ini=U0,rm=R,
                        frequencies_GHz=frequencies,B_FR=B_FR,cmb_model=cmb_label)

## 11. 多周波pixel RM復元: noiseless closureと現実的な困難の可視化

noiseless caseでは2周波間のphase差から \(R=\arg(P_1P_2^*)/[2(\lambda_1^2-\lambda_2^2)]\) を回収する。
phase wrappingの範囲を明示的に確認する。

beam/noise付きcaseでは各pixelでQ,Uを \(\lambda^2\) の一次式としてfitし、
\(R=\mathrm{Im}(P_*^* dP/d\lambda^2)/(2|P_*|^2)\) を計算。
同じbeamに揃えたQ/Uへ、pixel面積に応じたμK-arcmin white noiseを加える。
これは**small-angle・pixel fitの診断**で、統計的に最適なCMB EB推定量ではない。
低偏光pixelをmaskするので、この復元マップを主trade-off計算へそのまま流用しない。
ここで雑音が大きければ「CMB実験でFRが測れない」とも断定しない。

In [ ]:
if RUN_CMB_DEMO:
    phase_limit=np.pi/(2*abs(lambda2[0]-lambda2[-1]))
    if np.max(abs(R))>=phase_limit:
        raise ValueError('RM exceeds two-frequency phase-unwrapping range')
    rm_phase=np.angle(P_rot[0]*P_rot[-1].conj())/(2*(lambda2[0]-lambda2[-1]))
    valid=abs(P0)>1e-8*np.max(abs(P0))
    print('Noiseless phase RM maximum error:',np.max(abs(rm_phase[valid]-R[valid])),'rad/m²')
    assert np.allclose(rm_phase[valid],R[valid],rtol=1e-7,atol=1e-7)

    beam_sigma=np.deg2rad(CMB_BEAM_FWHM_ARCMIN/60)/np.sqrt(8*np.log(2))
    beam=np.exp(-0.5*LMAG**2*beam_sigma**2)
    pixel_arcmin=FOV_DEG*60/MAP_N
    noise_per_pixel=CMB_NOISE_UK_ARCMIN/pixel_arcmin
    rng=np.random.default_rng(SEED+701)
    P_noisy=np.array([np.fft.ifft2(np.fft.fft2(p)*beam) for p in P_rot])
    P_noisy += noise_per_pixel[:,None,None]*(rng.normal(size=P_noisy.shape)+1j*rng.normal(size=P_noisy.shape))
    weight=1/noise_per_pixel**2
    pivot=np.average(lambda2,weights=weight)
    design=np.column_stack([np.ones(len(lambda2)),lambda2-pivot])
    estimator=np.linalg.inv(design.T@(weight[:,None]*design))@design.T*weight[None,:]
    coeff=np.einsum('af,fxy->axy',estimator,P_noisy)
    intercept,slope=coeff
    rm_pixel=np.imag(intercept.conj()*slope)/(2*np.maximum(abs(intercept)**2,1e-30))
    # Diagnostic threshold based on true beam-smoothed polarization, explicitly an oracle mask.
    P_beam=np.fft.ifft2(np.fft.fft2(P0)*beam)
    mask=abs(P_beam)>np.percentile(abs(P_beam),30)
    print('Input Q/U noise per pixel [microK]:',noise_per_pixel)
    print('Diagnostic pixel RM residual rms:',np.std((rm_pixel-R)[mask]),'rad/m²')
    print('This value is not an EB reconstruction N_L or an experiment forecast.')
    fig,axes=plt.subplots(1,3,figsize=(15,4))
    for ax,field,title in zip(axes,[R,rm_phase,np.where(mask,rm_pixel,np.nan)],
                             ['input RM','noiseless phase RM','beam + noise pixel fit RM']):
        im=ax.imshow(field.T,origin='lower');ax.set_title(title);plt.colorbar(im,ax=ax,label='rad/m²')
    plt.tight_layout();plt.savefig(WORK_DIR/'cmb_rm_reconstruction_diagnostic.png',dpi=150);plt.show()

## 12. 検証・provenance・次の収束テスト

下の検証はsynthetic/TNG両方で実行。失敗を隠すための平均規格化・振幅の人工的な増幅はしない。
物理解析へ進む前に少なくとも:
- GRID_N、MAP_N、同じ観測filterを変えた収束確認（NGP/セルサイズ/視線step）。
- TNG300-3 → -2/-1。同じ恒星質量完全性を満たすサンプルで比較。
- coneのguard z幅・snapshot cadence・geometry seedの変更。
- 実銀河sampleのn(z)、bias、等級/色selection、photo-z PDF・外れ値・success rate。
- 複数の独立視野でconnected covarianceを評価。周期複製を独立なuniverseと数えない。
- CMB実験ごとのEB/multifrequency復元N_L、lens B-mode、Milky Way sky、maskを追加。
- ビン幅TRUE_Z_EDGESを変え、検出S/Nと同時振幅誤差の双方を比較。

既存correlation.ipynbは変更しない。出力はWORK_DIRへ保存。

In [ ]:
assert np.isfinite(rm_layers).all() and np.isfinite(X).all()
assert np.allclose(rphoto['contamination'].sum(1),1)
assert np.allclose(X_layers.sum(0),X,rtol=1e-9,atol=1e-10*max(1.,np.max(abs(X))))
if rphoto['fit']['full_rank']:
    assert np.allclose(rphoto['fit']['A'],1,atol=1e-6)
    assert np.allclose(rphoto['fit']['recovered_injection'],INJECTION_AMPLITUDES,atol=1e-6)
assert np.all(np.linalg.eigvalsh(rphoto['cov'])>0)
# Increasing independent RM noise must not improve precision for a fixed galaxy realization.
if len(RM_NOISE_RMS_GRID)>1:
    k0=('truth',0.,1.,float(min(RM_NOISE_RMS_GRID)))
    k1=('truth',0.,1.,float(max(RM_NOISE_RMS_GRID)))
    assert representative[k1]['fit']['snr_joint'] <= representative[k0]['fit']['snr_joint']*(1+1e-9)

config_names=['MODE','SIMULATION','TRUE_Z_EDGES','CONE_Z_RANGE','MAP_N','GRID_N','FOV_DEG',
    'MIN_STELLAR_MASS_MSUN','MIN_STAR_PARTICLES','USE_RSD','USE_CELL_HYDROGEN_FRACTION',
    'EXCLUDE_STAR_FORMING_GAS','SEED','RM_SMOOTH_FWHM_ARCMIN','RM_NOISE_RMS_GRID',
    'MW_RESIDUAL_RMS','EXTRA_PATH_RM_RMS','SURVEY_AREA_DEG2','N_BANDS','ELL_MAX',
    'Z_SIGMA0_GRID','KEEP_FRACTIONS','PHOTO_Z_BIAS','PHOTO_OUTLIER_FRACTION','SPEC_SIGMA0',
    'N_CATALOG_REALIZATIONS','CMB_E_MODEL','CMB_FREQUENCIES_GHZ','CMB_NOISE_UK_ARCMIN',
    'CMB_BEAM_FWHM_ARCMIN','CMB_CL_FILE','READ_BATCH','INJECTION_AMPLITUDES']
def jsonable(v):
    if isinstance(v,np.ndarray): return v.tolist()
    if isinstance(v,np.generic): return v.item()
    if isinstance(v,Path): return str(v)
    return v
provenance={k:jsonable(globals()[k]) for k in config_names}
provenance.update(base_path=str(BASE_PATH),box_mpc=box_mpc,h=h,
    actual_L=ell.tolist(),patch_modes=patch_modes.tolist(),cache_version=CACHE_VERSION,
    omega_m=float(cosmo.Om0),
    tiles=[{k:jsonable(v) for k,v in t.items()} for t in tiles],
    snapshots={str(t['snap']):t['snapshot_z'] for t in tiles},
    status='conditional Gaussian sensitivity study; not an end-to-end CMB forecast',
    versions={k:__import__(k).__version__ for k in ['numpy','scipy','pandas','h5py','astropy']})
(WORK_DIR/'run_configuration.json').write_text(json.dumps(provenance,indent=2),encoding='utf-8')
print('Checks passed. Outputs:',WORK_DIR.resolve())
display(pd.DataFrame([dict(file=p.name,KiB=round(p.stat().st_size/1024,1))
                      for p in sorted(WORK_DIR.iterdir()) if p.is_file()]))